# Module 2: RAG (Retrieval Augmented Generation) Applications
## Topics Covered
- Retrieval Augmented Generation (RAG) fundamentals
- Context-Aware AI & Real-Time Information Retrieval
- LangChain for RAG
- Gradio — Chatbot UI Development
- LlamaIndex — Knowledge Retrieval Pipelines

---
**Real-world scenario**: Building a company knowledge base chatbot that answers employee questions from internal HR documents and policies in real time.

In [ ]:
# Install required packages
%pip install -q langchain langchain-openai langchain-community langchain-chroma
%pip install -q chromadb tiktoken gradio
%pip install -q llama-index llama-index-llms-openai llama-index-embeddings-openai

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "your-api-key-here")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("Ready:", OPENAI_API_KEY != "your-api-key-here")

---
## 1. What is RAG?

**Retrieval Augmented Generation** solves the core LLM limitation: **hallucination and stale knowledge**.

```
User Query → Retriever (finds relevant docs) → Augmented Prompt → LLM → Grounded Answer
```

| Without RAG | With RAG |
|-------------|----------|
| Answers from training data only | Answers from YOUR documents |
| Knowledge cutoff date | Real-time / updated information |
| Can hallucinate facts | Grounded in retrieved context |
| No citations possible | Can cite source documents |

In [ ]:
# Simulate a company's HR documents
hr_documents = [
    """VACATION POLICY 2024
    Full-time employees receive 15 days of paid vacation per year.
    Part-time employees receive 7 days pro-rated based on hours worked.
    Vacation days must be approved by your manager at least 2 weeks in advance.
    Unused vacation days (up to 5) can be rolled over to the next year.
    Vacation requests during peak season (Dec 20 - Jan 3) require 4 weeks notice.""",
    
    """REMOTE WORK POLICY 2024
    Employees may work remotely up to 3 days per week with manager approval.
    Core hours for remote work are 10 AM - 3 PM in your local timezone.
    All remote work requires a secure VPN connection.
    Remote work is not permitted during client-facing weeks.
    Equipment allowance: $500 per year for home office setup.""",
    
    """HEALTH BENEFITS 2024
    Medical: Company covers 80% of premium for employee, 60% for dependents.
    Dental: Included with all full-time plans - up to $2000 per year.
    Vision: Eye exam covered annually, $200 frame allowance.
    Mental Health: 12 free therapy sessions per year via EAP.
    Wellness stipend: $300 per year for gym, apps, or wellness programs.""",
    
    """PERFORMANCE REVIEW PROCESS
    Performance reviews occur bi-annually: June and December.
    Self-assessments are due 2 weeks before the review date.
    Peer feedback is collected from 3-5 colleagues chosen by the employee.
    Ratings: Exceeds Expectations, Meets Expectations, Needs Improvement.
    Salary adjustments linked to December reviews only, effective January 1st.""",
    
    """PARENTAL LEAVE POLICY
    Primary caregiver: 16 weeks fully paid parental leave.
    Secondary caregiver: 6 weeks fully paid parental leave.
    Applies to birth, adoption, and foster placement.
    Leave can be taken within 12 months of child's arrival.
    Return-to-work program available with phased re-entry over 4 weeks."""
]

print(f"Loaded {len(hr_documents)} HR policy documents")

---
## 2. LangChain RAG Pipeline

**Pipeline steps**:
1. **Load** documents
2. **Split** into chunks
3. **Embed** chunks into vectors
4. **Store** in vector database
5. **Retrieve** relevant chunks for query
6. **Generate** answer with LLM

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain.schema import Document

# Step 1: Create Document objects
docs = [
    Document(page_content=doc, metadata={"source": f"hr_policy_{i+1}"})
    for i, doc in enumerate(hr_documents)
]

# Step 2: Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " "]
)
splits = text_splitter.split_documents(docs)
print(f"Split {len(docs)} docs into {len(splits)} chunks")

# Step 3 & 4: Embed and store in ChromaDB
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="hr_policies"
)
print("Vector store created!")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Step 5: Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Step 6: RAG Prompt
rag_prompt = ChatPromptTemplate.from_template("""
You are an HR assistant. Answer the employee's question based ONLY on the provided context.
If the answer is not in the context, say "I don't have that information in our policy documents."

Context:
{context}

Question: {question}

Answer:
""")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build RAG chain using LCEL
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Test queries
questions = [
    "How many vacation days do I get as a full-time employee?",
    "Can I work from home? What are the requirements?",
    "What maternity leave am I entitled to?",
    "How does the performance review process work?"
]

for q in questions:
    print(f"\n❓ {q}")
    answer = rag_chain.invoke(q)
    print(f"💬 {answer}")
    print("-" * 60)

---
## 3. Gradio Chatbot UI

**Gradio** makes it easy to create interactive web UIs for AI applications with just a few lines of Python.

In [ ]:
import gradio as gr

# Chat history maintained in Gradio state
def hr_chatbot(message, history):
    """HR Policy chatbot with RAG backing"""
    try:
        # Retrieve relevant docs
        relevant_docs = retriever.invoke(message)
        context = format_docs(relevant_docs)
        
        # Build prompt with chat history
        history_text = ""
        for human, ai in history[-3:]:  # Last 3 turns for context
            history_text += f"Human: {human}\nAssistant: {ai}\n"
        
        full_prompt = f"""You are an HR assistant. Use the context to answer questions.
        
Previous conversation:
{history_text}

Context from HR policies:
{context}

Current question: {message}
Answer:"""
        
        response = llm.invoke(full_prompt)
        
        # Add source attribution
        sources = list(set([doc.metadata.get('source', 'Unknown') for doc in relevant_docs]))
        source_note = f"\n\n*Sources: {', '.join(sources)}*"
        
        return response.content + source_note
    except Exception as e:
        return f"Error: {str(e)}"

# Create Gradio interface
demo = gr.ChatInterface(
    fn=hr_chatbot,
    title="🏢 HR Policy Assistant",
    description="Ask me anything about company policies: vacation, remote work, benefits, and more!",
    examples=[
        "How many vacation days do I get?",
        "What is the remote work policy?",
        "Tell me about parental leave benefits",
        "When are performance reviews conducted?"
    ],
    theme=gr.themes.Soft()
)

# Launch the app
# share=True creates a public link (useful for demos)
demo.launch(share=False, show_error=True)

---
## 4. LlamaIndex — Alternative RAG Framework

**LlamaIndex** (formerly GPT Index) specializes in RAG and knowledge retrieval with a simpler API for document indexing.

**LangChain vs LlamaIndex**:
| Feature | LangChain | LlamaIndex |
|---------|-----------|------------|
| Primary focus | General AI orchestration | RAG & knowledge retrieval |
| Learning curve | Steeper | Gentler for RAG |
| Flexibility | Higher | More opinionated |
| Index types | Basic | Rich (Tree, List, Knowledge Graph) |

In [ ]:
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.llms.openai import OpenAI as LlamaOpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

# Configure LlamaIndex settings
Settings.llm = LlamaOpenAI(model="gpt-4o-mini", temperature=0)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# Create LlamaIndex Documents
llama_docs = [
    Document(text=doc, metadata={"policy_type": ["vacation", "remote", "health", "performance", "parental"][i]})
    for i, doc in enumerate(hr_documents)
]

# Build Vector Index (automatic chunking + embedding)
index = VectorStoreIndex.from_documents(llama_docs, show_progress=True)
print("LlamaIndex index built!")

In [ ]:
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import VectorIndexRetriever

# Create query engine
query_engine = index.as_query_engine(
    similarity_top_k=3,
    response_mode="compact"  # 'compact', 'refine', 'tree_summarize'
)

# Query with source nodes
question = "What dental and vision benefits does the company offer?"
response = query_engine.query(question)

print(f"Question: {question}")
print(f"\nAnswer: {response}")
print(f"\nSource nodes used:")
for node in response.source_nodes:
    print(f"  - Score: {node.score:.3f} | Policy: {node.metadata.get('policy_type', 'N/A')}")

In [ ]:
# LlamaIndex Chat Engine - maintains conversation history
from llama_index.core.memory import ChatMemoryBuffer

chat_engine = index.as_chat_engine(
    chat_mode="condense_plus_context",
    memory=ChatMemoryBuffer.from_defaults(token_limit=3900),
    verbose=True
)

# Multi-turn conversation
print("=== Multi-turn HR Chat ===")
response1 = chat_engine.chat("How much vacation time do I get?")
print(f"Q1: How much vacation time do I get?")
print(f"A1: {response1}\n")

response2 = chat_engine.chat("Can I roll over unused days?")
print(f"Q2: Can I roll over unused days?")
print(f"A2: {response2}\n")

response3 = chat_engine.chat("What about requesting time off during Christmas?")
print(f"Q3: What about requesting time off during Christmas?")
print(f"A3: {response3}")

---
## Summary

| Component | Purpose | Library |
|-----------|---------|--------|
| Document Loader | Load source documents | LangChain/LlamaIndex |
| Text Splitter | Chunk documents optimally | RecursiveCharacterTextSplitter |
| Embeddings | Convert text to vectors | OpenAIEmbeddings / text-embedding-3-small |
| Vector Store | Store & search embeddings | ChromaDB / LlamaIndex VectorStoreIndex |
| Retriever | Find relevant chunks | k-NN similarity search |
| RAG Chain | End-to-end pipeline | LCEL (LangChain) |
| Chat UI | User interface | Gradio ChatInterface |
| Chat Memory | Multi-turn context | LlamaIndex ChatMemoryBuffer |

### Key Takeaways
- RAG grounds LLM answers in your documents, reducing hallucinations
- LangChain LCEL makes it easy to compose retrieval + generation pipelines
- Gradio provides instant UI for demo and production chatbots
- LlamaIndex offers simpler RAG-first APIs with built-in chat memory

### Next Steps
- Module 3: Vector Databases — deep dive into ChromaDB, embeddings, and semantic search